# 02 — Qwen3.5-4B NF4 + double quantisation on a Colab T4: speed + QLoRA fine-tuning

On the laptop CPU, bitsandbytes **dequantises every 4-bit weight to fp32 on every token**
(56% of decode time in the profile), so NF4-dq ran at **0.17 tok/s** vs Ollama's 1.92 tok/s.
On a GPU bitsandbytes has CUDA kernels that work on the packed 4-bit weights, and NF4 + double
quantisation is what makes **QLoRA** training fit in 15 GB.

**This notebook**
1. Check the GPU and install libraries
2. Load Qwen3.5-4B in NF4 + double quantisation and time it (same prompt as the laptop)
3. QLoRA fine-tune (LoRA adapters on the frozen NF4 base)
4. Compare answers before / after, save the adapter to Drive (and optionally the Hub)

Notebook **03** then serves the model + adapter with **vLLM** and measures throughput.

**How to run**
- *VS Code:* install the **Google Colab** extension → open this notebook → *Select Kernel* → *Colab* →
  new server → **GPU: T4**.
- *Browser:* colab.research.google.com → *Upload* this file → *Runtime → Change runtime type → T4 GPU*.

**T4 notes:** no bfloat16 → everything uses **fp16**. No FlashAttention-2 (Turing GPU).

## 1. GPU + install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Qwen3.5 needs a recent transformers. flash-linear-attention = Triton kernels for Qwen3.5's
# linear-attention layers (without it transformers uses a slower pure-PyTorch fallback).
!pip install -q -U "transformers>=5.2" accelerate bitsandbytes peft trl datasets
!pip install -q -U flash-linear-attention || echo "fla not installed - using the PyTorch fallback"

In [ ]:
import torch, transformers, bitsandbytes, peft, trl
print("torch", torch.__version__, "| transformers", transformers.__version__, "| bnb", bitsandbytes.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Save everything to Google Drive: free Colab sessions disconnect, and notebook 03 needs the adapter.
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/model_quantization"
import os; os.makedirs(WORK, exist_ok=True)
print("Saving to", WORK)

In [ ]:
# Hugging Face token: add it once under Colab's key icon (Secrets) as HF_TOKEN.
# Needed only for uploading; Qwen3.5-4B itself downloads without a token.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (fine unless you upload):", type(e).__name__)

## 2. Load NF4 + double quantisation and time it

Same settings as `src/model_quantization/quantize.py`, except **compute in fp16**: the project uses
bfloat16 on GPUs, but the T4 has no bfloat16 hardware.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,      # <- double quantisation
    bnb_4bit_compute_dtype=torch.float16,
)

import time
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                             dtype=torch.float16, device_map="cuda:0")
model.eval()
print(f"Loaded in {time.perf_counter() - t0:.0f} s | GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Weights: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
# Same CSV columns as the laptop API's logs/requests.csv, so the results can be compared directly.
import csv, time, uuid
from datetime import datetime
from pathlib import Path

LOG_CSV = Path(WORK) / "colab_requests.csv"
COLUMNS = ["request_id", "model", "model_name", "start_time", "end_time", "duration_sec", "duration_min",
           "prompt", "response", "prompt_tokens", "new_tokens", "tokens_per_second", "status", "error"]

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

def log_row(row):
    new = not LOG_CSV.exists()
    with LOG_CSV.open("a", newline="", encoding="utf-8-sig" if new else "utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS, quoting=csv.QUOTE_ALL, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in COLUMNS})

In [ ]:
def ask(prompt, max_new_tokens=200, label="nf4-dq-t4", m=None):
    """Generate one answer, time it and append it to the CSV log."""
    m = m or model
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(m.device)
    torch.cuda.synchronize(); start, t0 = now(), time.perf_counter()
    with torch.inference_mode():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize(); secs, end = time.perf_counter() - t0, now()
    new = out[0, inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(new, skip_special_tokens=True).strip()
    log_row(dict(request_id=uuid.uuid4().hex[:12], model=label, model_name=MODEL_ID, start_time=start,
                 end_time=end, duration_sec=round(secs, 2), duration_min=round(secs / 60, 3), prompt=prompt,
                 response=answer, prompt_tokens=inputs.input_ids.shape[1], new_tokens=len(new),
                 tokens_per_second=round(len(new) / secs, 2), status="ok"))
    return answer, secs, len(new)

ask("Hi", max_new_tokens=8)  # warm-up (CUDA kernels, Triton compilation)
PROMPT = "What is NF4 quantisation? One sentence."
answer, secs, n = ask(PROMPT, max_new_tokens=40)
print(answer)
print(f"\n{n} tokens in {secs:.2f} s = {n / secs:.2f} tok/s")

In [ ]:
import pandas as pd
pd.DataFrame([
    {"where": "Laptop CPU - Ollama qwen3.5:4b (Q4_K_M)", "duration_sec": 21.18, "tok/s": 1.92},
    {"where": "Laptop CPU - NF4-dq (bitsandbytes)",       "duration_sec": 236.29, "tok/s": 0.17},
    {"where": "Colab T4  - NF4-dq (bitsandbytes)",        "duration_sec": round(secs, 2), "tok/s": round(n / secs, 2)},
])

## 3. QLoRA fine-tuning

The NF4 + double-quantised base stays **frozen**; only small LoRA matrices (≈1% of the
parameters) are trained in fp16. `paged_adamw_8bit` is bitsandbytes' paged optimizer: optimizer state
moves to CPU RAM when GPU memory spikes (same "paging" idea as vLLM's KV cache).

The demo uses 1,000 examples of `yahma/alpaca-cleaned`. **To train on your own data**, replace the
dataset cell with a list of `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` rows.

In [ ]:
# Questions to compare before / after training
EVAL_PROMPTS = [
    "Give three tips for staying healthy.",
    "Explain what a neural network is to a 10-year-old.",
    "Write a haiku about quantisation.",
]
before = {p: ask(p, max_new_tokens=120, label="nf4-dq-t4-base")[0] for p in EVAL_PROMPTS}

In [ ]:
from datasets import load_dataset

def to_messages(ex):
    user = ex["instruction"] + (f"\n\n{ex['input']}" if ex["input"] else "")
    return {"messages": [{"role": "user", "content": user}, {"role": "assistant", "content": ex["output"]}]}

ds = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=42).select(range(1000))
ds = ds.map(to_messages, remove_columns=ds.column_names)
ds[0]["messages"]

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

args = SFTConfig(
    output_dir="/content/qlora-run",
    max_steps=60,                       # demo length; ~1 epoch of 1,000 examples is ~60 steps of 16
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=True, bf16=False,              # T4: fp16 only
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_length=512,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=lora, processing_class=tokenizer)

# PEFT creates the LoRA weights in bfloat16 for this model, but fp16 mixed precision on a T4 cannot
# unscale bf16 gradients ("_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16").
# Keep the trainable LoRA weights in fp32 (standard for QLoRA); the frozen NF4 base is unchanged.
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
trainer.model.print_trainable_parameters()

In [ ]:
t0 = time.perf_counter()
trainer.train()
print(f"Training took {(time.perf_counter() - t0) / 60:.1f} min | peak GPU memory "
      f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

## 4. Before / after, then save

In [ ]:
ft = trainer.model
ft.eval()
ft.config.use_cache = True
rows = []
for p in EVAL_PROMPTS:
    a, s, n = ask(p, max_new_tokens=120, label="nf4-dq-t4-qlora", m=ft)
    rows.append({"prompt": p, "before": before[p], "after": a, "after_sec": round(s, 2)})
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

In [ ]:
ADAPTER_DIR = f"{WORK}/qwen3.5-4b-nf4-dq-qlora-adapter"
ft.save_pretrained(ADAPTER_DIR)       # only the LoRA weights (tens of MB), not the 4B base
tokenizer.save_pretrained(ADAPTER_DIR)
!du -sh "{ADAPTER_DIR}"
print("CSV log:", LOG_CSV)

In [ ]:
# Optional: upload the adapter to the Hugging Face Hub (needs the HF_TOKEN secret with write access)
PUSH = False
HF_REPO = "PrabhuAIML/Qwen3.5-4B-NF4-DQ-QLoRA-alpaca"
if PUSH:
    ft.push_to_hub(HF_REPO, private=True)
    tokenizer.push_to_hub(HF_REPO, private=True)

**Next:** *Runtime → Restart session* (or disconnect) and open **03_colab_t4_vllm.ipynb**.
vLLM installs its own torch version, so it gets a fresh runtime.